# Segmentation with SAM3

## Attach GPU

Runtime in the taskbar

Change runtime type

Set Hardware Accelerator to T4

Save - *this will likely restart Colab Runtime*

## Setup Dataset

Open files in the sidebar

Right-click and add folders - *in this example we use "images" and "labels"*

Import images for annotation - *you may need to adjust names or do batches to avoid confusing clips and gestures*


## Installation

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e "."
!pip install -e ".[notebooks]"
%cd /content

## Import Libraries


In [ ]:
import os, sam3, glob, torch, torchvision

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Rectangle

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import plot_results

# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

## Request access to SAM3 & Login

To use SAM3, you need to request access by filling out this form on Hugging Face: https://huggingface.co/facebook/sam3

In [ ]:
from huggingface_hub import login
login()

## Initialize SAM3


In [ ]:
model_path = f"sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=model_path)
processor = Sam3Processor(model, confidence_threshold=0.5)

## Setup Dataset


In [ ]:
image_files = glob.glob("/content/images/*.png") #can be home, depending on which folder you're working from - try command !pwd to check
print(f"There are {len(image_files)} images available to annotate")

## Set Image & Generate masks with text prompt

In [ ]:
image_path = image_files[0]
image = Image.open(image_path)
inference_state = processor.set_image(image)

processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(state=inference_state, prompt="hand")

## Show the results

In [ ]:
plot_results(image, inference_state)

## Select & save the best Mask

In [ ]:
for i, (mask, box) in enumerate(zip(inference_state["masks"], inference_state["boxes"])):
  print(f"MASK PREDICTION {i}")
  mask = mask.cpu()[0] * 255
  x0, y0, x1, y1 = box.cpu()
  h = y1 - y0
  w = x1 - x0

  fig, ax = plt.subplots(1, 2)

  ax[0].imshow(image, cmap='gray')

  ax[0].add_patch(Rectangle((x0, y0), w, h,
             edgecolor = 'red',
             fill=False,
             lw=1))

  ax[1].imshow(mask, cmap='gray')
  plt.show()
  print()

In [ ]:
#Set prediction to save

SAVE = 0
if SAVE > len(inference_state["masks"])-1:
  print(f"invalid save request")
else:
  mask = inference_state["masks"][SAVE].cpu().numpy().astype(np.uint8)[0] * 255
  save_path = image_path.replace("images", "labels")
  Image.fromarray(mask).save(save_path)
  print(f"saved prediction number {SAVE} - {save_path}")


## Iterate entire dataset and save first prediction

In [ ]:
for i, image_path in enumerate(image_files):
  print(f"image {i} - {image_path}")
  image = Image.open(image_path)
  inference_state = processor.set_image(image)

  processor.reset_all_prompts(inference_state)
  inference_state = processor.set_text_prompt(state=inference_state, prompt="hand")
  #plot_results(image, inference_state)
  #plt.show()

  if (len(inference_state["masks"]) == 0):
    print(f"no masks found for {image_path}")
    mask = np.zeros((480, 640)).astype(np.uint8)
  else:
    mask = inference_state["masks"][SAVE].cpu().numpy().astype(np.uint8)[0] * 255

  save_path = image_path.replace("images", "labels")
  Image.fromarray(mask).save(save_path)
  print(f"saved prediction - {save_path}")
  print()

## Save files to your machine before exiting

Make sure naming conventions are reverted if you've made any changes

In [ ]:
!zip -r labels.zip labels/ #this converts the folder to a zip for a single download